# 基于 GAN 的人脸生成与性别控制系统

课程设计项目 | StyleGAN2-ADA (NVIDIA) + InterFaceGAN 潜空间操控

### 功能
- 随机生成：1024x1024 高保真人脸图像
- 性别控制：连续滑块调节男女性别属性
- 渐变 Morphing：同一身份从男性到女性的平滑过渡
- 截断分析：探索生成质量与多样性的 trade-off

## 1. 环境初始化

In [ ]:
# 在 import 官方模块之前禁用 CUDA 编译（纯 PyTorch 算子已验证正确）
from src import silence_ops

import os, torch, dnnlib, legacy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
device = 'cuda'
print(f'{torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}\n')

In [ ]:
# 加载预训练 StyleGAN2-ADA 模型（FFHQ, 1024x1024）
model_path = 'weights/ffhq.pkl'
if not os.path.exists(model_path):
    print('下载权重中 (~300MB)...')
    import requests, tqdm
    r = requests.get('https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl', stream=True)
    os.makedirs('weights', exist_ok=True)
    with open(model_path, 'wb') as f:
        for chunk in tqdm.tqdm(r.iter_content(8192), total=int(r.headers.get('content-length',0))//8192):
            f.write(chunk)

with dnnlib.util.open_url(model_path) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device).eval()

print(f'模型已加载: {G.img_resolution}x{G.img_resolution}, z_dim={G.z_dim}, W 层数={G.mapping.num_ws}')

In [ ]:
# 加载预计算的性别方向
if os.path.exists('weights/gender_dir_official.pt'):
    gender_dir = torch.load('weights/gender_dir_official.pt', map_location=device)
    print(f'性别方向已加载: {gender_dir.shape}')
else:
    torch.manual_seed(42)
    gender_dir = torch.randn(1, G.mapping.num_ws, G.z_dim, device=device)
    gender_dir = gender_dir / gender_dir.norm(p=2, dim=-1, keepdim=True)
    print('警告: 使用演示方向（运行 compute_gender.py 获取真实方向）')

## 2. 工具函数

In [ ]:
@torch.no_grad()
def gen(z=None, seed=None, psi=0.7):
    """生成人脸: z -> W -> 图像"""
    if seed is not None: torch.manual_seed(seed)
    if z is None: z = torch.randn(1, G.z_dim, device=device)
    ws = G.mapping(z, None, truncation_psi=psi)
    return G.synthesis(ws, noise_mode='const'), ws, z

def img2arr(t):
    """tensor [-1,1] -> numpy uint8 [0,255]"""
    return ((t.squeeze(0).permute(1,2,0).cpu().detach()+1)*127.5).clamp(0,255).to(torch.uint8).numpy()

def show(t, title='', s=6):
    plt.figure(figsize=(s,s)); plt.imshow(img2arr(t)); plt.axis('off')
    if title: plt.title(title, fontsize=13)
    plt.tight_layout(); plt.show()

def grid(imgs, titles=None, cols=4, fs=(16,12)):
    rows = (len(imgs)+cols-1)//cols
    fig, ax = plt.subplots(rows, cols, figsize=fs)
    ax = ax.flatten() if rows*cols>1 else [ax]
    for i, a in enumerate(ax):
        if i < len(imgs):
            a.imshow(img2arr(imgs[i]))
            if titles and i < len(titles): a.set_title(titles[i], fontsize=9)
        a.axis('off')
    plt.tight_layout(); plt.show()

print('就绪。')

## 3. 随机人脸生成

In [ ]:
print('随机生成 6 张人脸...\n')
imgs = [gen(seed=i)[0] for i in range(6)]
grid(imgs, [f'#{i+1}' for i in range(6)], cols=3, fs=(14,10))

## 4. 性别控制

原理 (InterFaceGAN, Shen et al. CVPR 2020)：StyleGAN2 的 W 空间中，语义属性对应线性方向。

$$W' = W + \alpha \cdot \mathbf{d}_{gender}$$

alpha > 0 偏向女性 | alpha < 0 偏向男性

In [ ]:
# 交互式性别滑块
torch.manual_seed(42)
base_ws = G.mapping(torch.randn(1, G.z_dim, device=device), None)

slider = widgets.FloatSlider(value=0, min=-5, max=5, step=0.2,
    description='性别:', style={'description_width':'50px'},
    layout=widgets.Layout(width='460px'))
label = widgets.Label(value='中性 (a=0)')
out = widgets.Output()

def update(change):
    a = change['new']
    if a > 0.5:
        label.value = f'女性 (a={a:+.1f})'
    elif a < -0.5:
        label.value = f'男性 (a={a:+.1f})'
    else:
        label.value = f'中性 (a={a:+.1f})'
    img = G.synthesis(base_ws + a * gender_dir, noise_mode='const')
    with out: clear_output(wait=True); show(img, label.value, s=7)

slider.observe(update, 'value')
update({'new':0})
display(widgets.VBox([slider, label, out]))

### 性别渐变 Morphing

In [ ]:
alphas = np.linspace(-5, 5, 11)
walk = [G.synthesis(base_ws + a*gender_dir, noise_mode='const') for a in alphas]
titles = [f'{"男" if a<-1 else "女" if a>1 else "中"} a={a:+.1f}' for a in alphas]
grid(walk, titles, cols=6, fs=(22,8))
print('男 <----------------------------> 女')

## 5. 多身份性别对比

In [ ]:
all_imgs, all_titles = [], []
for pid in range(6):
    torch.manual_seed(pid*13+7)
    ws = G.mapping(torch.randn(1, G.z_dim, device=device), None)
    for lbl, a in [('男', -4), ('中', 0), ('女', +4)]:
        all_imgs.append(G.synthesis(ws+a*gender_dir, noise_mode='const'))
        all_titles.append(f'ID#{pid+1} {lbl}')
grid(all_imgs, all_titles, cols=3, fs=(12,18))

## 6. 截断技巧 (Truncation Trick)

截断系数 psi 控制质量与多样性的权衡：w' = w_avg + psi * (w - w_avg)
- psi = 0：平均脸（质量最高，多样性为零）
- psi = 1：完全随机（多样性最高，可能出现伪影）

In [ ]:
torch.manual_seed(999)
z = torch.randn(1, G.z_dim, device=device)
trunc = [gen(z=z, psi=p)[0] for p in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]]
grid(trunc, [f'psi={p}' for p in [0.0,0.2,0.4,0.6,0.8,1.0]], cols=3, fs=(15,10))

## 7. 批量导出

In [ ]:
os.makedirs('outputs', exist_ok=True)
print('导出 12 张人脸...')
for i in range(12):
    img, _, _ = gen(seed=i*100)
    Image.fromarray(img2arr(img)).save(f'outputs/face_{i:03d}.png')
saved = [gen(seed=i*100)[0] for i in range(12)]
grid(saved, [f'#{i+1}' for i in range(12)], cols=4, fs=(18,14))
print('已保存到 outputs/')

## 8. 技术原理

### GAN 训练目标

$$\min_G \max_D \; \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

### StyleGAN2 核心创新 (Karras et al., CVPR 2020)

1. **映射网络** (z -> W)：将输入噪声映射到解缠的潜空间
2. **权重解调** (Weight Demodulation)：消除 StyleGAN1 中的水滴伪影
3. **噪声注入**：在每个分辨率层级添加随机细节
4. **路径长度正则化**：使 W 空间更平滑，插值更自然

### InterFaceGAN 潜空间操控 (Shen et al., CVPR 2020)

W 空间中的语义属性对应线性方向。通过训练线性 SVM 找到决策超平面，法向量即为操控方向。

$$\mathbf{w}' = \mathbf{w} + \alpha \cdot \mathbf{n}_{attribute}$$

适用于：性别、年龄、表情、姿态、发色、眼镜等多种属性。

## 9. 总结

| 指标 | 数值 |
|------|------|
| 输出分辨率 | 1024x1024 |
| 预训练数据 | FFHQ (7万张高清人脸) |
| 模型参数 | ~3000万 |
| 生成速度 | ~0.5秒/张 (RTX 4060) |
| 性别分类器 | CLIP ViT-B/32（零样本） |
| 方向方法 | W 空间线性 SVM |

导出 HTML：`jupyter nbconvert --to html --execute demo.ipynb`